In [1]:
import pandas as pd
import plotly_express as px
import streamlit as st
%matplotlib inline

In [2]:
df = pd.read_csv('vehicles_us.csv') #leer archivo
df.isna().sum()  # aqui determinamos los datos que nos hacen falta


price               0
model_year       3619
model               0
condition           0
cylinders        5260
fuel                0
odometer         7892
transmission        0
type                0
paint_color      9267
is_4wd          25953
date_posted         0
days_listed         0
dtype: int64

In [3]:
df['is_4wd'] = df['is_4wd'].fillna(0) # Deducimos que el dato faltante indica que no es un auto 4x4, rellenamos con 0.

df['paint_color'] = df['paint_color'].fillna('unknown') # Al ver que el color no esta ingresado, colocaremos que es dato desconocido de momento, y anotatlo para corregirlo en la base de datos.

df.isna().sum() # Comprobamos que los datos ha sido corregidos para la proyección.

price              0
model_year      3619
model              0
condition          0
cylinders       5260
fuel               0
odometer        7892
transmission       0
type               0
paint_color        0
is_4wd             0
date_posted        0
days_listed        0
dtype: int64

In [4]:
# rellenaremos con la mediana en las columnas de datos faltantes.
#  Calcular la mediana agrupada

mediana_años = df.groupby('model')['model_year'].transform('median')
# Usamos el fillna con lo que acabamos de crear.
df['model_year'] = df['model_year'].fillna(mediana_años)
df.isna().sum() # Comprobamos que los datos ha sido corregidos para la proyección.

price              0
model_year         0
model              0
condition          0
cylinders       5260
fuel               0
odometer        7892
transmission       0
type               0
paint_color        0
is_4wd             0
date_posted        0
days_listed        0
dtype: int64

In [5]:
# # Utilizaremos la media de los cilindros según el modelo para rellenar los huecos
mediana_cilindros = df.groupby('model')['cylinders'].transform('median')
df['cylinders'] = df['cylinders'].fillna(mediana_cilindros)
df.isna().sum() # Comprobamos que los datos ha sido corregidos para la proyección.

price              0
model_year         0
model              0
condition          0
cylinders          0
fuel               0
odometer        7892
transmission       0
type               0
paint_color        0
is_4wd             0
date_posted        0
days_listed        0
dtype: int64

In [6]:
# Utilizaremos la media de odometraje según el modelo para rellenar los huecos
# 3. El odómetro
mediana_odometro = df.groupby('model')['odometer'].transform('median')
df['odometer'] = df['odometer'].fillna(mediana_odometro)

# Verificamos que ya no haya ningún NaN en toda la tabla
print(df.isna().sum())

price            0
model_year       0
model            0
condition        0
cylinders        0
fuel             0
odometer        41
transmission     0
type             0
paint_color      0
is_4wd           0
date_posted      0
days_listed      0
dtype: int64


In [7]:
# ún quedan algunos huecos por lo que utilizaremos la mediana de odometraje para hacerlo
# 1. Calculamos la mediana global de todo el kilometraje
mediana_global_odometro = df['odometer'].median()

# 2. Rellenamos los 41 huecos restantes con ese valor general
df['odometer'] = df['odometer'].fillna(mediana_global_odometro)

# 3. Verificamos por última vez (¡ahora sí, puros ceros!)
print(df.isna().sum())

price           0
model_year      0
model           0
condition       0
cylinders       0
fuel            0
odometer        0
transmission    0
type            0
paint_color     0
is_4wd          0
date_posted     0
days_listed     0
dtype: int64


In [8]:

#  gráfico de dispersión

fig = px.scatter(df, 
                 x='odometer', 
                 y='price', 
                 title='Relación entre Precio y Kilometraje',
                 color='condition',
                 opacity=0.4)


fig.show()

Vemos una grafica con algunos datos extremos, ya que si bien podriamos tene autos de super lujo, es probable que en el kilometraje haya algunos datos mal capturados.

In [9]:
#  Creamos un DataFrame temporal solo para la gráfica, quitando los autos extremadamente caros o con kilometraje irreal
df_grafica = df[(df['price'] < 80000) & (df['odometer'] < 300000)]


fig = px.scatter(df_grafica, 
                 x='odometer', 
                 y='price', 
                 color='condition', # Separamos por color
                 opacity=0.4,       # Le damos transparencia a los puntos
                 title='Precio vs. Kilometraje por Condición (Sin Outliers)')

fig.show()

In [10]:
df.to_csv('vehicles_us_limpio.csv', index=False)